# Pipeline Avançado: Ensemble de Seleção + Ensemble de Modelos

**Etapa 0** — Baseline (todos os features, sem seleção)  
**Etapa 1** — Ensemble de Seleção de Features (voting / ranking / stability)  
**Etapa 2** — Ensemble de Modelos por dataset (Voting soft+hard · Stacking com OOF)

Modelos base: RandomForest · XGBoost · DecisionTree · GaussianNB (calibrado) · KNN · GradientBoosting  
Meta-modelo (Stacking): LogisticRegression treinado sobre OOF predictions

In [ ]:
# ── Padrão ───────────────────────────────────────────────────────
import warnings
from collections import defaultdict, Counter
from pathlib import Path
from statistics import mode as stat_mode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier, StackingClassifier, VotingClassifier,
)
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss,
    matthews_corrcoef, precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

# ── Optuna ───────────────────────────────────────────────────────
import optuna

# ── XGBoost (opcional) ───────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost não disponível. Instale com: pip install xgboost")

# ── mRMR (opcional) ──────────────────────────────────────────────
try:
    from mrmr import mrmr_classif
    MRMR_AVAILABLE = True
except ImportError:
    MRMR_AVAILABLE = False
    print("⚠️  mRMR não disponível. Instale com: pip install mrmr-selection")

# ── Configurações globais ─────────────────────────────────────────
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
print("✅ Imports carregados.")


## Pré-processamento

IQR capping → remoção de colunas irrelevantes → encode do label.

In [ ]:
DATA_PATH = Path("../data/processed/DoS_with_gap_features.csv")
df = pd.read_csv(DATA_PATH)
print(f"Shape original: {df.shape}")

def cap_outliers_iqr(df, columns):
    df_out = df.copy()
    for col in columns:
        q1, q3 = df_out[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        df_out[col] = df_out[col].clip(lower=q1 - 1.5*iqr, upper=q3 + 1.5*iqr)
    return df_out

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_tratado = cap_outliers_iqr(df, numeric_cols)

COLUNAS_REMOVER = [
    "frame.time_delta_displayed","frame.time_epoch","frame.time_invalid",
    "frame.time_relative","tcp.srcport","tcp.dstport",
    "frame.coloring_rule.name","frame.coloring_rule.string",
    "frame.comment","frame.comment.expert","frame.encap_type",
    "frame.file_off","frame.ignored","frame.incomplete",
    "frame.interface_id","frame.interface_name","frame.link_nr",
    "frame.marked","frame.md5_hash","frame.number","frame.offset_shift",
    "mqtt.msgid","mqtt.username","mqtt.passwd","mqtt.willmsg","mqtt.willtopic",
]

num_df = df_tratado.select_dtypes(include=["int64","float64"])
X = num_df.drop(columns=[c for c in COLUNAS_REMOVER if c in num_df.columns], errors="ignore")
X = X.fillna(0).replace([np.inf, -np.inf], 0)

le = LabelEncoder()
y = le.fit_transform(df_tratado["type"])

print(f"Features: {X.shape[1]} | Amostras: {X.shape[0]}")
print(f"Classes: {le.classes_}  →  {dict(enumerate(le.classes_))}")


## Etapa 0 — Baseline (todos os features, sem seleção)

Split 70/30 estratificado. Sem Optuna — hiperparâmetros razoáveis fixos.  
Serve como referência para comparar o ganho das etapas seguintes.

In [ ]:
# ── Split 70/30 ──────────────────────────────────────────────────
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

def _metrics(model, X_te, y_te, name):
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te) if hasattr(model, "predict_proba") else None
    ll = log_loss(y_te, y_proba) if y_proba is not None else np.nan
    return {
        "model":     name,
        "accuracy":  round(accuracy_score(y_te, y_pred), 5),
        "precision": round(precision_score(y_te, y_pred, average="macro", zero_division=0), 5),
        "recall":    round(recall_score(y_te, y_pred, average="macro", zero_division=0), 5),
        "f1_macro":  round(f1_score(y_te, y_pred, average="macro", zero_division=0), 5),
        "mcc":       round(matthews_corrcoef(y_te, y_pred), 5),
        "log_loss":  round(ll, 5) if np.isfinite(ll) else np.nan,
    }

BASELINE_MODELS = {
    "RandomForest":     RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":          XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                      verbosity=0, eval_metric="logloss") if XGBOOST_AVAILABLE
                        else RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "DecisionTree":     DecisionTreeClassifier(random_state=42),
    "GaussianNB_Cal":   CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5),
    "KNN":              KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    # HistGBT: mesma precisao que GradientBoosting, 10-50x mais rapido
    "GradientBoosting": HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1,
                                                       max_depth=4, random_state=42),
}

print("Treinando modelos baseline…")
baseline_rows, baseline_trained = [], {}
for name, model in BASELINE_MODELS.items():
    model.fit(X_tr_b, y_tr_b)
    baseline_trained[name] = model
    row = _metrics(model, X_te_b, y_te_b, name)
    baseline_rows.append(row)
    print(f"  ✓ {name:20s}  F1={row['f1_macro']:.4f}  LogLoss={row['log_loss']}")

# ── Voting soft + hard ───────────────────────────────────────────
vc_s = VotingClassifier(list(baseline_trained.items()), voting="soft", n_jobs=-1)
vc_s.fit(X_tr_b, y_tr_b)
vc_h = VotingClassifier(list(baseline_trained.items()), voting="hard", n_jobs=-1)
vc_h.fit(X_tr_b, y_tr_b)
baseline_rows += [_metrics(vc_s, X_te_b, y_te_b, "Voting_Soft"),
                  _metrics(vc_h, X_te_b, y_te_b, "Voting_Hard")]

# ── Stacking com OOF ─────────────────────────────────────────────
stk_base = [(n, clone(m)) for n, m in BASELINE_MODELS.items()]
# cv=3 suficiente para OOF estavel, 40%% mais rapido que cv=5
stk = StackingClassifier(estimators=stk_base,
                         final_estimator=LogisticRegression(max_iter=1000, C=1.0),
                         cv=3, passthrough=False, n_jobs=-1)
stk.fit(X_tr_b, y_tr_b)
baseline_rows.append(_metrics(stk, X_te_b, y_te_b, "Stacking_OOF"))

df_baseline = pd.DataFrame(baseline_rows).set_index("model")
print("\n── Métricas Etapa 0 — Baseline ──────────────────────────")
display(df_baseline)

# ── Gráfico ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ENSEMBLE_NAMES = {"Voting_Soft", "Voting_Hard", "Stacking_OOF"}
colors_b = ["#1D4ED8" if m in ENSEMBLE_NAMES else "#93C5FD" for m in df_baseline.index]
for ax, col, lbl in zip(axes, ["f1_macro","mcc","log_loss"],
                                ["F1 Macro","MCC","Log Loss ↓"]):
    bars = ax.barh(df_baseline.index, df_baseline[col], color=colors_b, edgecolor="white")
    ax.invert_yaxis()
    ax.set_title(lbl, fontweight="bold")
    for bar, v in zip(bars, df_baseline[col]):
        ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
                f"{v:.4f}" if np.isfinite(v) else "N/A", va="center", fontsize=8.5)
plt.suptitle("Etapa 0 — Baseline (todos os features)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

best_b = df_baseline["f1_macro"].idxmax()
print(f"\n🏆 Melhor baseline: {best_b}  F1={df_baseline.loc[best_b,'f1_macro']:.5f}")


## Etapa 1 — Ensemble de Seleção de Features

Cada seletor é aplicado sobre **todo o X** (sem CV — apenas para geração de datasets).  
Agregação por três estratégias:

| Estratégia | Descrição |
|---|---|
| **Voting count** | Nº de seletores que escolheram a feature |
| **Mean rank** | Posição média nos rankings de cada seletor (menor = melhor) |
| **Consensus majority** | Features em ≥ metade dos seletores |

Datasets gerados: um por seletor individual + `consensus_majority` + `consensus_top15_rank`.

In [ ]:
# ── Parâmetros de seleção ─────────────────────────────────────────
K_MRMR=K_FISHER=K_PEARSON=K_TREE=K_SVC=K_LASSO = 15

# ── Fisher Score ─────────────────────────────────────────────────
def fisher_score(X, y):
    classes      = np.unique(y)
    overall_mean = X.mean(axis=0).values
    scores       = np.zeros(X.shape[1])
    for i in range(X.shape[1]):
        vals = X.iloc[:, i].values
        bet, wit = 0.0, 0.0
        for c in classes:
            mask = y == c; nc = mask.sum()
            if nc > 0:
                mu_c = vals[mask].mean()
                bet += nc * (mu_c - overall_mean[i])**2
                wit += nc * vals[mask].var()
        scores[i] = bet / wit if wit > 1e-10 else 0.0
    return scores

# ── Funções de seleção ───────────────────────────────────────────
def _selector_functions():
    def sel_mrmr(X_tr, y_tr):
        if not MRMR_AVAILABLE: return []
        try: return mrmr_classif(X_tr, y_tr, K=min(K_MRMR, X_tr.shape[1]))
        except Exception as e: print(f"[mRMR] {e}"); return []

    def sel_fisher(X_tr, y_tr):
        sc = fisher_score(X_tr, y_tr)
        return (pd.DataFrame({"f": X_tr.columns,"s": sc})
                .sort_values("s", ascending=False)
                .head(min(K_FISHER, len(X_tr.columns)))["f"].tolist())

    def sel_pearson(X_tr, y_tr):
        ac = X_tr.apply(lambda c: abs(np.corrcoef(c, y_tr)[0,1])).fillna(0)
        return ac.sort_values(ascending=False).head(min(K_PEARSON,len(ac))).index.tolist()

    def sel_extratrees(X_tr, y_tr):
        et = ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        et.fit(X_tr, y_tr)
        imp = pd.Series(et.feature_importances_, index=X_tr.columns)
        return imp.sort_values(ascending=False).head(min(K_TREE,X_tr.shape[1])).index.tolist()

    def sel_svc(X_tr, y_tr):
        Xs = StandardScaler().fit_transform(X_tr)
        svc = LinearSVC(C=0.1, penalty="l1", dual=False, max_iter=5000, random_state=42)
        svc.fit(Xs, y_tr)
        coef = np.mean(np.abs(svc.coef_),axis=0) if svc.coef_.ndim>1 else np.abs(svc.coef_[0])
        return pd.Series(coef, index=X_tr.columns).sort_values(ascending=False).head(min(K_SVC,X_tr.shape[1])).index.tolist()

    def sel_lasso(X_tr, y_tr):
        Xs = StandardScaler().fit_transform(X_tr)
        lasso = LassoCV(cv=3, random_state=42, max_iter=10000, n_jobs=-1)
        lasso.fit(Xs, y_tr)
        return pd.Series(np.abs(lasso.coef_), index=X_tr.columns).sort_values(ascending=False).head(min(K_LASSO,X_tr.shape[1])).index.tolist()

    def sel_lowvar(X_tr, y_tr):
        sel = VarianceThreshold(threshold=0.0); sel.fit(X_tr)
        return X_tr.columns[sel.get_support(indices=True)].tolist()

    return {"mRMR":sel_mrmr,"Fisher":sel_fisher,"Pearson":sel_pearson,
            "ExtraTrees":sel_extratrees,"LinearSVC_L1":sel_svc,
            "LassoCV":sel_lasso,"LowVariance":sel_lowvar}

selector_funcs = _selector_functions()
print("Seletores:", list(selector_funcs.keys()))

# ── Aplica todos os seletores sobre X completo ───────────────────
print("\nAplicando seletores sobre X completo…")
raw_selections = {}   # {selector: [ranked feature list]}
for sel_name, sel_fn in selector_funcs.items():
    cols = sel_fn(X, y)
    raw_selections[sel_name] = cols
    print(f"  {sel_name:15s} → {len(cols)} features")

# ── Tabela de rankings (posição de cada feature por seletor) ──────
all_features = X.columns.tolist()
n_sel = len(raw_selections)
PENALTY = max(len(v) for v in raw_selections.values()) + 1  # rank para não-selecionadas

rank_data = {}
for sel_name, cols in raw_selections.items():
    ranks = {}
    for pos, feat in enumerate(cols):
        ranks[feat] = pos + 1          # rank 1-based (1 = melhor)
    for feat in all_features:
        if feat not in ranks:
            ranks[feat] = PENALTY      # não selecionada → penalidade
    rank_data[sel_name] = ranks

df_ranks = pd.DataFrame(rank_data).loc[all_features]  # features × seletores

# ── Métricas de agregação ─────────────────────────────────────────
df_ranks["voting_count"] = (df_ranks < PENALTY).sum(axis=1)
df_ranks["mean_rank"]    = df_ranks[list(raw_selections.keys())].mean(axis=1)
df_ranks = df_ranks.sort_values(["voting_count","mean_rank"], ascending=[False,True])

print("\n── Top 20 features por consenso ─────────────────────────")
display(df_ranks[["voting_count","mean_rank"]].head(20))

# ── Gera os datasets ──────────────────────────────────────────────
selector_datasets = {}

# Um dataset por seletor individual
for sel_name, cols in raw_selections.items():
    if cols:
        selector_datasets[sel_name] = X[cols].copy()

# Consensus: majority (>= metade dos seletores)
majority_thresh = n_sel // 2
consensus_maj_cols = df_ranks[df_ranks["voting_count"] >= majority_thresh].index.tolist()
selector_datasets["consensus_majority"] = X[consensus_maj_cols].copy()

# Consensus: top 15 por mean_rank
consensus_rank_cols = df_ranks.sort_values("mean_rank").head(15).index.tolist()
selector_datasets["consensus_top15_rank"] = X[consensus_rank_cols].copy()

print(f"\nDatasets gerados ({len(selector_datasets)}):")
for k, v in selector_datasets.items():
    print(f"  {k:25s} → {v.shape[1]} features: {v.columns.tolist()}")


### Visualizações — Ensemble de Seleção

In [ ]:
# ── 1. Heatmap: seletores × features (top 20 por voting_count) ──
top20 = df_ranks.head(20).index.tolist()
heat_data = df_ranks.loc[top20, list(raw_selections.keys())]
# Mascara: 1 se selecionada (rank < PENALTY), 0 caso contrário
heat_binary = (heat_data < PENALTY).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap de presença (binário)
sns.heatmap(heat_binary.T, annot=True, fmt="d", cmap="Blues",
            cbar=False, linewidths=0.5, ax=axes[0])
axes[0].set_title("Seletores × Features Top-20\n(1 = selecionada)", fontweight="bold")
axes[0].set_xlabel("Feature"); axes[0].set_ylabel("Seletor")
axes[0].tick_params(axis='x', rotation=45)

# Bar chart: voting count para top 20
vc = df_ranks.loc[top20, "voting_count"]
colors_vc = ["#1D4ED8" if v == n_sel else "#93C5FD" if v >= majority_thresh else "#E2E8F0"
             for v in vc]
axes[1].barh(top20, vc, color=colors_vc, edgecolor="white")
axes[1].invert_yaxis()
axes[1].axvline(majority_thresh, color="#F59E0B", linewidth=1.5, linestyle="--",
                label=f"Limiar majoritário ({majority_thresh})")
axes[1].set_xlabel("Nº de seletores que escolheram a feature")
axes[1].set_title("Voting Count — Top 20 features", fontweight="bold")
axes[1].legend(); 
for i, v in enumerate(vc):
    axes[1].text(v+0.05, i, str(int(v)), va="center", fontsize=9)

plt.tight_layout(); plt.show()

# ── 2. Mean rank por feature (top 20) ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
mr = df_ranks.loc[top20, "mean_rank"].sort_values()
ax.barh(mr.index, mr.values, color="#3B82F6", edgecolor="white")
ax.invert_yaxis()
ax.set_xlabel("Mean rank (menor = mais importante)")
ax.set_title("Mean Rank — Top 20 features", fontweight="bold")
for i, v in enumerate(mr.values):
    ax.text(v+0.1, i, f"{v:.1f}", va="center", fontsize=8.5)
plt.tight_layout(); plt.show()


## Etapa 2 — Ensemble de Modelos com Optuna (por dataset de features)

Para cada dataset da Etapa 1, o pipeline executa:

1. **Optuna** (paralelo, 1 thread por modelo) — tuna RF, XGBoost, DT, GNB, KNN, GB
2. **VotingClassifier Soft** — média de probabilidades
3. **VotingClassifier Hard** — votação majoritária
4. **StackingClassifier** — base models com OOF predictions → meta-model LogisticRegression

CV externo: 5-fold estratificado. CV interno (Optuna): 3-fold sobre `X_tr`.

In [ ]:
def _to_array(X):
    return X.values if isinstance(X, pd.DataFrame) else X

# ── RandomForest ─────────────────────────────────────────────────
def _obj_rf(trial, X_tr, y_tr):
    p = {"n_estimators": trial.suggest_int("n_estimators", 50, 400, step=50),
         "max_depth":    trial.suggest_categorical("max_depth", [None, 5, 10, 20, 30]),
         "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
         "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
         "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
         "bootstrap":    trial.suggest_categorical("bootstrap", [True, False]),
         "random_state": 42, "n_jobs": -1}
    return cross_val_score(RandomForestClassifier(**p), _to_array(X_tr), y_tr,
                           cv=3, scoring="f1_macro", n_jobs=-1).mean()

# ── XGBoost ──────────────────────────────────────────────────────
def _obj_xgb(trial, X_tr, y_tr):
    p = {"n_estimators":     trial.suggest_int("n_estimators", 50, 400, step=50),
         "max_depth":        trial.suggest_int("max_depth", 2, 10),
         "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
         "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
         "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
         "gamma":            trial.suggest_float("gamma", 0.0, 1.0),
         "reg_alpha":        trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
         "reg_lambda":       trial.suggest_float("reg_lambda", 1e-5, 1.0, log=True),
         "random_state": 42, "n_jobs": -1,
         "verbosity": 0, "eval_metric": "logloss"}
    mdl = XGBClassifier(**p) if XGBOOST_AVAILABLE else RandomForestClassifier(random_state=42, n_jobs=-1)
    return cross_val_score(mdl, _to_array(X_tr), y_tr,
                           cv=3, scoring="f1_macro", n_jobs=-1).mean()

# ── DecisionTree ─────────────────────────────────────────────────
def _obj_dt(trial, X_tr, y_tr):
    p = {"max_depth":    trial.suggest_categorical("max_depth", [None, 3, 5, 10, 20]),
         "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
         "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 15),
         "criterion":    trial.suggest_categorical("criterion", ["gini", "entropy"]),
         "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
         "splitter":     trial.suggest_categorical("splitter", ["best", "random"]),
         "random_state": 42}
    return cross_val_score(DecisionTreeClassifier(**p), _to_array(X_tr), y_tr,
                           cv=3, scoring="f1_macro", n_jobs=-1).mean()

# ── GaussianNB calibrado ─────────────────────────────────────────
def _obj_gnb(trial, X_tr, y_tr):
    vs  = trial.suggest_float("var_smoothing", 1e-11, 1e-5, log=True)
    cm  = trial.suggest_categorical("calibration_method", ["sigmoid", "isotonic"])
    mdl = CalibratedClassifierCV(GaussianNB(var_smoothing=vs), method=cm, cv=3)
    return cross_val_score(mdl, _to_array(X_tr), y_tr,
                           cv=3, scoring="f1_macro", n_jobs=-1).mean()

def _build_gnb(**p):
    return CalibratedClassifierCV(GaussianNB(var_smoothing=p["var_smoothing"]),
                                  method=p.get("calibration_method","isotonic"), cv=3)

# ── KNN ──────────────────────────────────────────────────────────
def _obj_knn(trial, X_tr, y_tr):
    p = {"n_neighbors": trial.suggest_int("n_neighbors", 1, 50),
         "weights":     trial.suggest_categorical("weights", ["uniform","distance"]),
         "metric":      trial.suggest_categorical("metric", ["euclidean","manhattan","minkowski"]),
         "p":           trial.suggest_int("p", 1, 3), "n_jobs": -1}
    return cross_val_score(KNeighborsClassifier(**p), _to_array(X_tr), y_tr,
                           cv=3, scoring="f1_macro", n_jobs=-1).mean()

# ── GradientBoosting ─────────────────────────────────────────────
def _obj_gb(trial, X_tr, y_tr):
    p = {"n_estimators":     trial.suggest_int("n_estimators", 50, 300, step=50),
         "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
         "max_depth":        trial.suggest_int("max_depth", 2, 8),
         "min_samples_split":trial.suggest_int("min_samples_split", 2, 20),
         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
         "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
         "random_state": 42}
    return cross_val_score(GradientBoostingClassifier(**p), _to_array(X_tr), y_tr,
                           cv=3, scoring="f1_macro", n_jobs=-1).mean()

# ── Mapa de modelos ───────────────────────────────────────────────
MODEL_OBJECTIVES = {
    "RandomForest":     (_obj_rf,  RandomForestClassifier,     {"random_state":42,"n_jobs":-1}),
    "XGBoost":          (_obj_xgb, XGBClassifier if XGBOOST_AVAILABLE else RandomForestClassifier,
                         {"random_state":42,"n_jobs":-1,"verbosity":0,"eval_metric":"logloss"}
                         if XGBOOST_AVAILABLE else {"random_state":42,"n_jobs":-1}),
    "DecisionTree":     (_obj_dt,  DecisionTreeClassifier,     {"random_state":42}),
    "GaussianNB_Cal":   (_obj_gnb, _build_gnb,                 {}),
    "KNN":              (_obj_knn, KNeighborsClassifier,        {"n_jobs":-1}),
    "GradientBoosting": (_obj_gb,  HistGradientBoostingClassifier, {"random_state":42}),
}
print("Modelos Optuna:", list(MODEL_OBJECTIVES.keys()))


### Pipeline Core — Tuning + Voting + Stacking

In [ ]:
# ── Worker Optuna (1 modelo, chamado em paralelo) ────────────────
def _tune_one(name, obj_fn, constructor, fixed, X_arr, y_tr, n_trials):
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )
    study.optimize(lambda t: obj_fn(t, X_arr, y_tr),
                   n_trials=n_trials, show_progress_bar=False)
    params = {**study.best_params, **fixed}
    return name, constructor(**params), params.copy()


def tune_fold(X_tr, y_tr, n_trials=30):
    """Tuna todos os modelos em paralelo via threading."""
    X_arr = _to_array(X_tr)
    results = Parallel(n_jobs=len(MODEL_OBJECTIVES), backend="threading")(
        delayed(_tune_one)(name, obj, ctor, fixed, X_arr, y_tr, n_trials)
        for name, (obj, ctor, fixed) in MODEL_OBJECTIVES.items()
    )
    instances   = {n: m for n, m, _ in results}
    best_params = {n: p for n, _, p in results}
    return instances, best_params


def _agg_params(plist):
    if not plist: return {}
    agg = {}
    for key in plist[0]:
        vals = [p[key] for p in plist if p.get(key) is not None]
        if not vals: agg[key] = None
        elif all(isinstance(v, float) for v in vals): agg[key] = float(np.mean(vals))
        elif all(isinstance(v, int)   for v in vals): agg[key] = int(round(np.mean(vals)))
        else:
            raw = stat_mode(str(v) for v in vals)
            agg[key] = {"None":None,"True":True,"False":False}.get(raw, raw)
    return agg


def _compute_metrics(model, X_val, y_val):
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val) if hasattr(model,"predict_proba") else None
    if y_proba is None and hasattr(model,"decision_function"):
        ds = model.decision_function(X_val)
        if ds.ndim == 1: ds = np.vstack([-ds, ds]).T
        y_proba = np.exp(ds) / np.exp(ds).sum(axis=1, keepdims=True)
    ll = np.nan
    if y_proba is not None:
        try: ll = log_loss(y_val, y_proba)
        except: pass
    return {"accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred, average="macro", zero_division=0),
            "recall":    recall_score(y_val, y_pred, average="macro", zero_division=0),
            "f1":        f1_score(y_val, y_pred, average="macro", zero_division=0),
            "mcc":       matthews_corrcoef(y_val, y_pred),
            "logloss":   ll}


def _build_fresh(name, params):
    """Cria instância nova (não treinada) do modelo com os best_params do fold."""
    _, constructor, _ = MODEL_OBJECTIVES[name]
    return constructor(**params)


def evaluate_dataset_cv(ds_name, X_ds, y, cv_splits=5, n_optuna_trials=30):
    """
    CV estratificado para um dataset de features.
    Por fold: Optuna → treina individuais → Voting soft/hard → Stacking OOF.
    Retorna: df_metrics (mean/std por modelo), best_hyperparams dict
    """
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    all_m  = defaultdict(lambda: defaultdict(list))
    fold_hp = defaultdict(list)

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_ds, y), start=1):
        print(f"  [Fold {fold_idx}/{cv_splits}] {ds_name}…")
        X_tr, X_val = X_ds.iloc[tr_idx], X_ds.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # ── Optuna ──────────────────────────────────────────────
        instances, bp_fold = tune_fold(X_tr, y_tr, n_trials=n_optuna_trials)
        for nm, p in bp_fold.items():
            fold_hp[nm].append(p)

        # ── Treina individuais ───────────────────────────────────
        trained = {}
        for nm, mdl in instances.items():
            mdl.fit(X_tr, y_tr)
            trained[nm] = mdl
            for k, v in _compute_metrics(mdl, X_val, y_val).items():
                all_m[nm][k].append(v)

        # ── Voting Soft ──────────────────────────────────────────
        vc_s_est = [(nm, _build_fresh(nm, bp_fold[nm])) for nm in trained]
        try:
            vc_s = VotingClassifier(vc_s_est, voting="soft", n_jobs=-1)
            vc_s.fit(X_tr, y_tr)
            for k, v in _compute_metrics(vc_s, X_val, y_val).items():
                all_m["Voting_Soft"][k].append(v)
        except Exception as e:
            print(f"    ⚠ Voting soft falhou: {e}")

        # ── Voting Hard ──────────────────────────────────────────
        vc_h_est = [(nm, _build_fresh(nm, bp_fold[nm])) for nm in trained]
        vc_h = VotingClassifier(vc_h_est, voting="hard", n_jobs=-1)
        vc_h.fit(X_tr, y_tr)
        for k, v in _compute_metrics(vc_h, X_val, y_val).items():
            all_m["Voting_Hard"][k].append(v)

        # ── Stacking OOF ─────────────────────────────────────────
        # Base models com params do fold (clones não treinados para OOF interno)
        stk_est = [(nm, _build_fresh(nm, bp_fold[nm])) for nm in trained]
        stk = StackingClassifier(
            estimators=stk_est,
            final_estimator=LogisticRegression(max_iter=1000, C=1.0, random_state=42),
            cv=3, passthrough=False, n_jobs=-1,  # cv=3 suficiente, 40%% mais rapido
        )
        stk.fit(X_tr, y_tr)
        for k, v in _compute_metrics(stk, X_val, y_val).items():
            all_m["Stacking_OOF"][k].append(v)

    # ── Agrega métricas ──────────────────────────────────────────
    rows = []
    for model_name, md in all_m.items():
        row = {"dataset": ds_name, "model": model_name}
        for metric, vals in md.items():
            finite = [v for v in vals if np.isfinite(v)]
            row[f"{metric}_mean"] = float(np.mean(finite)) if finite else np.nan
            row[f"{metric}_std"]  = float(np.std(finite))  if finite else np.nan
        rows.append(row)

    best_hp = {nm: _agg_params(plist) for nm, plist in fold_hp.items()}
    return pd.DataFrame(rows), best_hp

print("✅ Pipeline core definido.")


### Orquestração, Visualização e Comparação Final

In [ ]:
# ── Gráfico por dataset ───────────────────────────────────────────
def plot_dataset_results(df_res, title):
    if df_res.empty: return
    METRICS = [("accuracy_mean","Accuracy"),("precision_mean","Precision"),
               ("recall_mean","Recall"),("f1_mean","F1 Macro"),
               ("mcc_mean","MCC"),("logloss_mean","Log Loss ↓")]
    ENSEMBLE = {"Voting_Soft","Voting_Hard","Stacking_OOF"}
    colors = ["#1D4ED8" if m in ENSEMBLE else "#93C5FD" for m in df_res["model"]]
    ncols,nrows = 3, int(np.ceil(len(METRICS)/3))
    fig, axes = plt.subplots(nrows, ncols, figsize=(17, 4.5*nrows))
    axes = axes.flatten()
    for ax,(metric,label) in zip(axes, METRICS):
        errs = df_res[metric.replace("_mean","_std")]
        bars = ax.barh(df_res["model"], df_res[metric], xerr=errs,
                       color=colors, capsize=4, edgecolor="white")
        ax.invert_yaxis(); ax.set_title(label, fontweight="bold")
        for bar, val in zip(bars, df_res[metric]):
            ax.text(bar.get_width()+0.004, bar.get_y()+bar.get_height()/2,
                    f"{val:.3f}" if np.isfinite(val) else "N/A", va="center", fontsize=8.5)
    for j in range(len(METRICS), len(axes)): axes[j].axis("off")
    plt.suptitle(title, fontsize=13, fontweight="bold"); plt.tight_layout(); plt.show()


# ── Heatmap comparativo ───────────────────────────────────────────
def plot_heatmap(df_all, metric="f1_mean", title_suffix=""):
    if df_all.empty: return
    pivot = df_all.pivot_table(index="dataset", columns="model", values=metric)
    fig, ax = plt.subplots(figsize=(max(12, len(pivot.columns)*1.8+1), len(pivot)*0.9+1.5))
    im = ax.imshow(pivot.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(pivot.columns))); ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right", fontsize=9)
    ax.set_yticklabels(pivot.index, fontsize=9)
    gmax = np.nanmax(pivot.values)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i,j]
            if np.isfinite(val):
                color = "white" if val > 0.75*gmax else "black"
                if val == gmax:
                    ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,
                                               linewidth=2.5,edgecolor="#F59E0B",facecolor="none"))
                ax.text(j,i,f"{val:.3f}",ha="center",va="center",
                        color=color,fontsize=8.5,fontweight="bold" if val==gmax else "normal")
    plt.colorbar(im, ax=ax, label=metric)
    ax.set_title(f"Dataset × Modelo — {metric}{title_suffix}  (⭐ = melhor global)",
                 fontsize=11, fontweight="bold", pad=10)
    plt.tight_layout(); plt.show()


# ── Melhor par (dataset × modelo) ─────────────────────────────────
def find_best_combo(df_all, hyperparams_all, metric="f1_mean"):
    ENSEMBLE = {"Voting_Soft","Voting_Hard","Stacking_OOF"}
    results = {}
    # Melhor modelo individual
    df_ind = df_all[~df_all["model"].isin(ENSEMBLE)]
    if not df_ind.empty:
        best = df_ind.sort_values(metric, ascending=False).iloc[0]
        results["Melhor individual"] = best
    # Melhor ensemble
    df_ens = df_all[df_all["model"].isin(ENSEMBLE)]
    if not df_ens.empty:
        best_e = df_ens.sort_values(metric, ascending=False).iloc[0]
        results["Melhor ensemble"] = best_e

    print("\n" + "★"*70)
    print(f"  RESULTADO FINAL — métrica: {metric}")
    print("★"*70)
    for label, best in results.items():
        ds, mod = best["dataset"], best["model"]
        params = hyperparams_all.get(ds, {}).get(mod, {})
        print(f"\n  [{label}]")
        print(f"    Dataset (seletor) : {ds}")
        print(f"    Modelo            : {mod}")
        print(f"    {metric:<20}  {best[metric]:.5f}  (±{best[metric.replace('_mean','_std')]:.5f})")
        print(f"    Todas as métricas:")
        for col in df_all.columns:
            if col.endswith("_mean") and col not in ("dataset","model"):
                v = best[col]
                print(f"      {col[:-5]:<12} {v:.5f}" if np.isfinite(v) else f"      {col[:-5]:<12} N/A")
        if params:
            print(f"    Hiperparâmetros (agregados entre folds):")
            for pk, pv in params.items():
                print(f"      {pk:<28} {pv}")
    print("\n" + "★"*70)
    return results


# ── Tabela de hiperparâmetros ──────────────────────────────────────
def summarize_hyperparams(hyperparams_all):
    rows = [{"dataset":ds,"model":mod,"param":pk,"value":pv}
            for ds,mods in hyperparams_all.items()
            for mod,params in mods.items()
            for pk,pv in params.items()]
    return pd.DataFrame(rows)


# ── Executa todos os datasets ─────────────────────────────────────
def run_all_datasets(selector_datasets, y, cv_splits=5, n_optuna_trials=30):
    df_all         = pd.DataFrame()
    hyperparams_all = {}

    for ds_name, X_ds in selector_datasets.items():
        print(f"\n{'═'*65}")
        print(f" Dataset: {ds_name}  ({X_ds.shape[1]} features)")
        print(f"{'═'*65}")

        df_res, best_hp = evaluate_dataset_cv(
            ds_name, X_ds, y,
            cv_splits=cv_splits, n_optuna_trials=n_optuna_trials,
        )
        display(df_res)
        plot_dataset_results(df_res, f"{ds_name} — Resultados CV")

        df_all = pd.concat([df_all, df_res], ignore_index=True)
        hyperparams_all[ds_name] = best_hp

    return df_all, hyperparams_all

print("✅ Orquestração definida.")


## Execução

Configure os parâmetros abaixo e rode a célula. Os resultados de cada dataset aparecem incrementalmente.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                   CONFIGURAÇÃO                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
CV_SPLITS        = 5
N_OPTUNA_TRIALS  = 20  # TPE converge com 20 trials; 30 era excessivo
RANK_METRIC      = "f1_mean"

# Pasta para salvar CSVs dos datasets de features (None = não salva)
OUTPUT_DIR = None   # ex: "../data/processed/feature_sets"

# ── Executa Etapa 2 em todos os datasets ─────────────────────────
df_all, hyperparams_all = run_all_datasets(
    selector_datasets,
    y,
    cv_splits=CV_SPLITS,
    n_optuna_trials=N_OPTUNA_TRIALS,
)

# ── Heatmap comparativo global ────────────────────────────────────
print("\n" + "─"*65)
print(" COMPARAÇÃO GLOBAL: datasets × modelos")
print("─"*65)
plot_heatmap(df_all, metric=RANK_METRIC)

# ── Comparação Etapa 0 vs Etapa 2 (VotingClassifier) ─────────────
# Coleta o melhor resultado de cada etapa para o F1 do ensemble
best_etapa0_f1 = df_baseline["f1_macro"].max()
best_etapa2 = df_all[df_all["model"].isin({"Voting_Soft","Voting_Hard","Stacking_OOF"})]
if not best_etapa2.empty:
    best_etapa2_f1 = best_etapa2["f1_mean"].max()
    delta = best_etapa2_f1 - best_etapa0_f1
    print(f"\n📊 Ganho Etapa 2 vs Baseline:")
    print(f"   Baseline melhor ensemble F1  : {best_etapa0_f1:.5f}")
    print(f"   Etapa 2  melhor ensemble F1  : {best_etapa2_f1:.5f}")
    print(f"   Delta                        : {delta:+.5f}")

# ── Resultado final ───────────────────────────────────────────────
best_combo = find_best_combo(df_all, hyperparams_all, metric=RANK_METRIC)

# ── Tabela de hiperparâmetros ─────────────────────────────────────
print("\n── Hiperparâmetros agregados (seletor × modelo) ────────────")
display(summarize_hyperparams(hyperparams_all))

# ── Salva datasets se solicitado ─────────────────────────────────
if OUTPUT_DIR is not None:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    for ds_name, X_ds in selector_datasets.items():
        path = Path(OUTPUT_DIR) / f"features_{ds_name}.csv"
        X_ds.to_csv(path, index=False)
        print(f"  Salvo: {path}")
